# cuDF — Basic Uses

Assumes cuDF is already installed. Same API as pandas, just runs on the GPU.

What's in here:
1. Build a DataFrame, filter, group, sum
2. Join two tables (`merge`)
3. Hop between cuDF and pandas
4. `cudf.pandas` — zero-code-change pandas acceleration
5. Single-GPU cuDF vs multi-GPU Dask-cuDF

In [1]:
import cudf
import cupy as cp

print(f"GPU: {cp.cuda.runtime.getDeviceProperties(0)['name'].decode()}")

# tiny GPU fleet inventory — one row per card, vram in GB
df = cudf.DataFrame({
    "gpu":  ["H100", "A100", "L40S", "L4", "T4", "H100", "A100", "L40S"],
    "vram": [80,    80,     48,     24,   16,   80,     80,     48],
})
df

GPU: NVIDIA GB10


,gpu,vram
0,H100,80
1,A100,80
2,L40S,48
3,L4,24
4,T4,16
5,H100,80
6,A100,80
7,L40S,48


In [2]:
# keep the datacenter cards (32GB+) and total VRAM per model
df[df["vram"] >= 32].groupby("gpu")["vram"].sum()

gpu
H100    160
A100    160
L40S     96
Name: vram, dtype: int64

## Joins

`merge` is the same as pandas — `on`, `how`, all of it. Just on the GPU.

In [3]:
# lookup table: GPU -> architecture
arch = cudf.DataFrame({
    "gpu":  ["H100", "A100", "L40S", "L4", "T4", "V100", "RTX_PRO_6000"],
    "arch": ["Hopper", "Ampere", "Ada", "Ada", "Turing", "Volta", "Blackwell"],
})
df.merge(arch, on="gpu", how="left")

,gpu,vram,arch
0,H100,80,Hopper
1,A100,80,Ampere
2,L40S,48,Ada
3,L4,24,Ada
4,T4,16,Turing
5,H100,80,Hopper
6,A100,80,Ampere
7,L40S,48,Ada


## Pandas interop

cuDF and pandas convert back and forth easily. Handy when you need to hand data to libraries that don't run on GPU yet (matplotlib, sklearn, etc.).

In [4]:
pdf = df.to_pandas()           # GPU -> CPU
back = cudf.from_pandas(pdf)   # back on the GPU
type(back), type(pdf)

(cudf.core.dataframe.DataFrame, pandas.core.frame.DataFrame)

## `cudf.pandas` — zero-code-change pandas on the GPU

Load the extension once at the top of a notebook (or run a script with `python -m cudf.pandas script.py`) and every `import pandas` automatically runs on the GPU. Falls back to CPU for anything cuDF doesn't support yet, so existing pandas code just works.

Has to be the **very first cell** to actually take effect — that's why this is shown as a snippet, not a runnable cell.

```python
%load_ext cudf.pandas
import pandas as pd  # this is now GPU-backed pandas

df = pd.read_parquet("gpu_fleet.parquet")
df.groupby("gpu")["vram"].sum()
```

## cuDF vs Dask-cuDF

Both use the same pandas-style API. Dask-cuDF just wraps cuDF and splits the data across multiple GPUs.

- **cuDF**: single GPU, eager (returns immediately), data has to fit in one GPU's VRAM. Good for low-latency work.
- **Dask-cuDF**: multi-GPU / multi-node, lazy (call `.compute()` to actually run it), partitions across GPUs. Use when your data is too big for one GPU.

In [5]:
import dask_cudf

# wrap the cuDF DataFrame as a 2-partition Dask-cuDF DataFrame
ddf = dask_cudf.from_cudf(df, npartitions=2)

# same API, but lazy — needs .compute() to actually run
ddf.groupby("gpu")["vram"].sum().compute()

gpu
A100    160
H100    160
L4       24
T4       16
L40S     96
Name: vram, dtype: int64